In [1]:
# Scenario: Customer Support Chatbot Workflow
# Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

# - State Definition (BotState)
# - The chatbot keeps track of:
# - The question asked by the customer.
# - The answer generated.
# - The history of all past questions.
# - Think of this as the chatbot’s notebook where it records the conversation.

# - Nodes (Functions)
# - get_answer:
# When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
# It also adds the question to the history log.
# - format_output:
# Before sending the response back to the customer, the chatbot reformats it into a friendly style:
# “Bot says: Answer to: What are your store hours?”

# - Graph Flow
# - The chatbot starts at the get_answer node (entry point).
# - Once the answer is generated, it flows to the format_output node.
# - Finally, the conversation ends at END, meaning the chatbot has
#  delivered its response.


from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


In [12]:
# ==========================================
# INTERACTIVE CUSTOMER SUPPORT CHATBOT
# ==========================================

import requests
from google.colab import userdata

# 🔑 Load API Key from Colab Secrets
API_KEY = userdata.get('Groq_api')

if not API_KEY:
    raise ValueError("❌ Add 'groq_api_key' in Colab Secrets (🔑 icon in sidebar).")

print("✅ Groq API key loaded.\n")


# ---------------- STATE ---------------- #
class ChatState:
    def __init__(self):
        self.question = ""
        self.answer = ""
        self.history = []


# ---------------- LLM CALL ---------------- #
def call_groq_llm(question, history):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful and friendly customer support agent for TechStore, "
                "an online electronics retailer. Answer questions about orders, products, "
                "returns, shipping, and warranties. Be concise, polite, and professional. "
                "Keep replies under 4 sentences."
            )
        }
    ]

    # Inject full conversation history so bot remembers context
    for q, a in history:
        messages.append({"role": "user", "content": q})
        messages.append({"role": "assistant", "content": a})

    messages.append({"role": "user", "content": question})

    payload = {
        "model": "llama-3.3-70b-versatile",
        "messages": messages,
        "temperature": 0.7,
        "max_tokens": 300
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"❌ Error calling API: {e}"


# ---------------- GRAPH NODES ---------------- #

def input_node(state):
    user_input = input("👤 You  : ")
    state.question = user_input.strip()
    return state

def answer_node(state):
    state.answer = call_groq_llm(state.question, state.history)
    return state

def history_node(state):
    state.history.append((state.question, state.answer))
    return state

def output_node(state):
    print(f"🤖 Bot  : {state.answer}")
    print("-" * 65)
    return state


# ---------------- MAIN CHAT LOOP ---------------- #

def run_chatbot():
    print("=" * 65)
    print("   💬 TechStore Customer Support — Live Chat")
    print("   Type 'exit' or 'quit' to end the session")
    print("=" * 65 + "\n")

    state = ChatState()

    while True:

        # Step 1: Take user input
        state = input_node(state)

        # Step 2: Check exit condition
        if state.question.lower() in ["exit", "quit", "bye"]:
            print("\n🤖 Bot  : Thank you for contacting TechStore support. Have a great day! 👋")
            print("=" * 65)
            print(f"📝 Session Summary: {len(state.history)} question(s) answered.")
            break

        # Step 3: Skip empty input
        if not state.question:
            print("⚠️  Please type a question.\n")
            continue

        # Step 4: Generate answer → Save history → Print reply
        state = answer_node(state)
        state = history_node(state)
        state = output_node(state)


# ---------------- RUN ---------------- #
if __name__ == "__main__":
    run_chatbot()

"""
How it works:
👤 You  : Hi, I ordered a laptop 3 days ago. When will it arrive?
🤖 Bot  : Typically orders arrive within 5-7 business days...
-----------------------------------------------------------------
👤 You  : What if it arrives damaged?
🤖 Bot  : Please contact us within 48 hours with photos...
-----------------------------------------------------------------
👤 You  : exit
🤖 Bot  : Thank you for contacting TechStore. Have a great day! 👋
"""

✅ Groq API key loaded.

   💬 TechStore Customer Support — Live Chat
   Type 'exit' or 'quit' to end the session

👤 You  : Hi, I ordered a laptop 3 days ago. When will it arrive?
🤖 Bot  : Thank you for shopping with TechStore. Your order is being processed, and you can expect to receive it within 5-7 business days. You can also track the status of your order by logging into your account on our website and clicking on "Order History".
-----------------------------------------------------------------
👤 You  : What if it arrives damaged?
🤖 Bot  : If your laptop arrives damaged, please contact us within 3 days of delivery and we'll be happy to assist you with a replacement or refund. You can reach us through our website's support page or by calling our customer service number. We'll also provide a return shipping label at no additional cost to you.
-----------------------------------------------------------------
👤 You  : okay no probelm 
🤖 Bot  : You're welcome, if you have any other quest

'\nHow it works:\n👤 You  : Hi, I ordered a laptop 3 days ago. When will it arrive?\n🤖 Bot  : Typically orders arrive within 5-7 business days...\n-----------------------------------------------------------------\n👤 You  : What if it arrives damaged?\n🤖 Bot  : Please contact us within 48 hours with photos...\n-----------------------------------------------------------------\n👤 You  : exit\n🤖 Bot  : Thank you for contacting TechStore. Have a great day! 👋\n'

In [5]:
# Scenario: Customer Support Call Center
# A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

# 1. State Definition (SupportState)
# The chatbot keeps track of:
# - query → What the customer asked.
# - category → Which department it belongs to (billing, tech, general).
# - response → What the bot replies with.
# Think of this as the customer’s “ticket form.”

# 2. Router Node (route_query)
# When a customer types a question, the router scans it:
# - If the query mentions “bill” or “payment”, it routes to billing_agent.
# - If it mentions “error” or “bug”, it routes to tech_agent.
# - Otherwise, it defaults to general_agent.
# This is like a receptionist deciding which desk you should go to.

# 3. Agent Nodes
# - billing_agent → Replies with “Billing dept: [query]”.
# - tech_agent → Replies with “Tech support: [query]”.
# - general_agent → Replies with “General help: [query]”.
# Each agent specializes in its own type of problem.

# 4. Graph Flow
# - Entry point: router.
# - Router decides the path based on the query.
# - The query flows into the correct agent node.
# - The agent generates a response and ends the conversation.


from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)


Scenario: AI-Powered Study Assistant (Flashcard-Based)
1. State Definition
The assistant maintains a notebook-like state for each learner:
- topic → The subject the learner is studying (e.g., "Photosynthesis").
- flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
- progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

2. Workflow (Graph of States)
Each learner interaction flows through nodes until a flashcard is delivered:
- Input Node
- Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
- State updates: topic = "cell biology"
- Generation Node (Groq API)
- Groq generates a flashcard:
- flashcard.question = "What organelle is known as the powerhouse of the cell?"
- flashcard.answer = "Mitochondria"
- Response Node
- Assistant presents the flashcard question to the learner.
- Evaluation Node
- Learner responds with their answer.
- Assistant checks correctness and updates progress.
- History Node
- Logs the flashcard attempt:
- progress = [{question, learner_answer, correct/incorrect}]

In [14]:
# ==========================================
# AI STUDY ASSISTANT (FLASHCARD - GROQ API)
# ==========================================

import requests
from google.colab import userdata

# 🔑 Load API Key from Colab Secrets
API_KEY = userdata.get('Groq_api')

if not API_KEY:
    raise ValueError("❌ Add 'groq_api_key' in Colab Secrets (🔑 icon in sidebar).")

print("✅ Groq API key loaded.\n")


# ---------------- STATE ---------------- #
class StudyState:
    def __init__(self):
        self.topic = ""
        self.flashcard = {"question": "", "answer": ""}
        self.progress = []  # list of {question, learner_answer, result}


# ---------------- LLM CALL ---------------- #
def call_groq_flashcard(topic):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    prompt = f"""
Generate ONE flashcard for studying: {topic}

Strictly use this format and nothing else:
Question: <write the question here>
Answer: <write the answer here>
"""

    payload = {
        "model": "llama-3.3-70b-versatile",   # ✅ Fixed: updated model name
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
        "max_tokens": 300
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]

        # Parse Question and Answer
        if "Question:" in content and "Answer:" in content:
            q = content.split("Question:")[1].split("Answer:")[0].strip()
            a = content.split("Answer:")[1].strip()
            return {"question": q, "answer": a}
        else:
            return {"question": "Could not parse question.", "answer": "Could not parse answer."}

    except Exception as e:
        return {"question": "API Error", "answer": str(e)}


# ---------------- GRAPH NODES ---------------- #

# Node 1: Input Topic
def input_node(state):
    state.topic = input("\n📘 Enter topic (or type 'exit'): ").strip()
    return state


# Node 2: Generate Flashcard
def generation_node(state):
    print("⏳ Generating flashcard...")
    state.flashcard = call_groq_flashcard(state.topic)
    return state


# Node 3: Show Question
def response_node(state):
    print(f"\n🧠 Flashcard Question:\n👉 {state.flashcard['question']}")
    return state


# Node 4: Evaluate Answer
def evaluation_node(state):
    # ✅ Fixed: guard against API error before asking answer
    if state.flashcard["question"] == "API Error":
        print(f"⚠️  Skipping evaluation due to API error: {state.flashcard['answer']}")
        return state

    learner_answer = input("\n✏️  Your Answer: ").strip()
    correct_answer = state.flashcard["answer"]

    # Simple evaluation (case-insensitive match)
    if learner_answer.lower() in correct_answer.lower():
        result = "Correct ✅"
    else:
        result = f"Incorrect ❌\n   ✔ Correct Answer: {correct_answer}"

    print(f"\n📊 Result: {result}")

    # Save progress
    state.progress.append({
        "question": state.flashcard["question"],
        "learner_answer": learner_answer,
        "result": result
    })

    return state


# Node 5: Show Progress
def history_node(state):
    if not state.progress:
        return state

    print("\n📈 Progress so far:")
    print("-" * 55)
    for i, entry in enumerate(state.progress, 1):
        print(f"{i}. Q : {entry['question']}")
        print(f"   You: {entry['learner_answer']}")
        print(f"   {entry['result']}")
        print()
    return state


# ---------------- GRAPH EXECUTION ---------------- #
def run_study_assistant():
    print("=" * 55)
    print("   🎓 AI Study Assistant — Flashcard Mode")
    print("   Type 'exit' to quit anytime")
    print("=" * 55)

    state = StudyState()

    while True:
        # Flow: Input → Generate → Ask → Evaluate → Progress

        state = input_node(state)

        if state.topic.lower() in ["exit", "quit", "bye"]:
            correct = sum(1 for p in state.progress if "✅" in p["result"])
            total   = len(state.progress)
            print(f"\n👋 Session ended!")
            print(f"🏆 Final Score: {correct}/{total} correct")
            print("=" * 55)
            break

        if not state.topic:
            print("⚠️  Topic cannot be empty. Please try again.")
            continue

        state = generation_node(state)
        state = response_node(state)
        state = evaluation_node(state)
        state = history_node(state)


# ---------------- MAIN ---------------- #
if __name__ == "__main__":
    run_study_assistant()

✅ Groq API key loaded.

   🎓 AI Study Assistant — Flashcard Mode
   Type 'exit' to quit anytime

📘 Enter topic (or type 'exit'): state
⏳ Generating flashcard...

🧠 Flashcard Question:
👉 What is a state in the context of government and politics?

✏️  Your Answer: i don't know 

📊 Result: Incorrect ❌
   ✔ Correct Answer: A state is a political entity with a defined territory, population, and government, exercising sovereignty over its territory and having the power to make and enforce laws.

📈 Progress so far:
-------------------------------------------------------
1. Q : What is a state in the context of government and politics?
   You: i don't know
   Incorrect ❌
   ✔ Correct Answer: A state is a political entity with a defined territory, population, and government, exercising sovereignty over its territory and having the power to make and enforce laws.


📘 Enter topic (or type 'exit'): exit

👋 Session ended!
🏆 Final Score: 0/1 correct


Scenario: AI-Powered Project Tracker (Task-Based)
1. State Definition
The assistant maintains a notebook-like state for each project:
- task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
- status → The current state of the task (e.g., "in progress", "completed", "blocked").
- log → A history of all updates, including who made them and when.

2. Workflow (Graph of States)
Each project update flows through nodes until the task status is refreshed:
- Input Node
- Team member submits an update (e.g., "Report draft completed").
- State updates: task = "Q1 financial report"
- Processing Node (Groq API)
- Groq interprets the update and assigns a status:
- status = "completed"
- Response Node
- Assistant confirms the update back to the team:
- "Task Q1 financial report marked as completed."
- History Node
- Logs the update:
- log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]

In [16]:
# ==========================================
# AI PROJECT TRACKER (FIXED + ROBUST VERSION)
# ==========================================

import requests
from datetime import datetime

# 🔑 Replace with your real Groq API Key
API_KEY = "Groq_api"


# ---------------- STATE ---------------- #
class ProjectState:
    def __init__(self):
        self.task = ""
        self.status = ""
        self.log = []


# ---------------- FALLBACK LOGIC ---------------- #
def fallback_status(update):
    update = update.lower()
    if "complete" in update or "done" in update:
        return "completed"
    elif "block" in update or "issue" in update:
        return "blocked"
    else:
        return "in progress"


# ---------------- LLM CALL ---------------- #
def call_groq_status(update_text):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    prompt = f"""
    Classify the task update into ONE word:
    - completed
    - in progress
    - blocked

    Update: {update_text}
    """

    payload = {
        "model": "llama3-70b-8192",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0
    }

    try:
        response = requests.post(url, headers=headers, json=payload)

        # 🔍 Debug (optional)
        # print("DEBUG:", response.text)

        if response.status_code == 200:
            result = response.json()["choices"][0]["message"]["content"].strip().lower()

            # Safety check
            if result in ["completed", "in progress", "blocked"]:
                return result
            else:
                return fallback_status(update_text)

        else:
            print("⚠️ API Error:", response.text)
            return fallback_status(update_text)

    except Exception as e:
        print("⚠️ Exception:", e)
        return fallback_status(update_text)


# ---------------- GRAPH NODES ---------------- #

# Node 1: Input
def input_node(state):
    state.task = input("\n📌 Enter task name (or 'exit'): ")

    if state.task.lower() == "exit":
        return state

    state.update = input("📝 Enter update: ")
    state.user = input("👤 Enter your name: ")

    return state


# Node 2: Process
def processing_node(state):
    state.status = call_groq_status(state.update)
    return state


# Node 3: Response
def response_node(state):
    print(f"\n🤖 Task '{state.task}' marked as: {state.status.upper()}")
    return state


# Node 4: History
def history_node(state):
    entry = {
        "task": state.task,
        "update": state.update,
        "status": state.status,
        "user": state.user,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    state.log.append(entry)
    return state


# Node 5: Show Log
def show_log_node(state):
    print("\n📊 Project Log:")
    for i, entry in enumerate(state.log, 1):
        print(f"{i}. [{entry['timestamp']}] {entry['user']}")
        print(f"   Task: {entry['task']}")
        print(f"   Update: {entry['update']}")
        print(f"   Status: {entry['status']}\n")
    return state


# ---------------- MAIN LOOP ---------------- #
def run_project_tracker():
    print("📁 AI Project Tracker (type 'exit' to quit)")

    state = ProjectState()

    while True:
        state = input_node(state)

        if state.task.lower() == "exit":
            print("👋 Exiting Project Tracker.")
            break

        state = processing_node(state)
        state = response_node(state)
        state = history_node(state)
        state = show_log_node(state)


# ---------------- RUN ---------------- #
if __name__ == "__main__":
    run_project_tracker()

📁 AI Project Tracker (type 'exit' to quit)

📌 Enter task name (or 'exit'): MERN stack project development
📝 Enter update: Frontend completed
👤 Enter your name: What work are you doing?
⚠️ API Error: {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}


🤖 Task 'MERN stack project development' marked as: COMPLETED

📊 Project Log:
1. [2026-03-20 06:15:00] What work are you doing?
   Task: MERN stack project development
   Update: Frontend completed
   Status: completed


📌 Enter task name (or 'exit'): exit
👋 Exiting Project Tracker.


Scenario: AI Symptom Tracker (Question-Based)
1. State Definition
The assistant maintains a notebook-like state for each patient:
- symptom → The patient’s reported issue (e.g., "persistent cough").
- observations → Notes or snippets generated by Groq about possible causes or related conditions.
- analysis → A synthesized interpretation of the observations.
- recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
- steps_done → A counter tracking progress through the workflow.

2. Workflow (Graph of States)
Each patient query flows through nodes:
- Symptom Input Node
- Patient reports a symptom.
- State updates: symptom = "persistent cough"
- Observation Node (Groq API)
- Groq generates possible related factors or general information.
- Updates observations.
- Analysis Node
- Joins observations and extracts key insights.
- Updates analysis.
- Conditional Node
- If fewer than 3 observations are collected → loop back to Observation Node.
- If 3+ observations are available → move to Recommendation Node.
- Recommendation Node
- Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
- Updates recommendation.
- End Node
- Delivers the final recommendation to the patient.

In [20]:
# ==========================================
# AI SYMPTOM TRACKER (SAFE + ROBUST VERSION)
# ==========================================

import requests
from google.colab import userdata

# 🔑 Load API Key from Colab Secrets
API_KEY = userdata.get('Groq_api')

if not API_KEY:
    raise ValueError("❌ Add 'groq_api_key' in Colab Secrets (🔑 icon in sidebar).")

print("✅ Groq API key loaded.\n")


# ---------------- STATE ---------------- #
class SymptomState:
    def __init__(self):
        self.symptom = ""
        self.observations = []
        self.analysis = ""
        self.recommendation = ""
        self.steps_done = 0


# ---------------- FALLBACK LOGIC ---------------- #
def fallback_observation(symptom):
    symptom = symptom.lower()
    if "cough" in symptom:
        return "Persistent cough may be related to infection, allergies, or irritation."
    elif "fever" in symptom:
        return "Fever can indicate infection or inflammation."
    elif "headache" in symptom:
        return "Headache may be due to stress, dehydration, or lack of sleep."
    elif "cold" in symptom:
        return "Cold symptoms are commonly caused by viral infections."
    elif "fatigue" in symptom:
        return "Fatigue may result from lack of sleep, stress, or underlying illness."
    else:
        return "Symptom may have multiple general causes; monitoring is advised."


def fallback_recommendation(symptom):
    return f"For '{symptom}', monitor your condition and consult a doctor if symptoms persist or worsen."


# ---------------- LLM CALL ---------------- #
def call_groq(prompt):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "llama-3.3-70b-versatile",   # ✅ Fixed: updated model
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a general health information assistant. "
                    "You do NOT diagnose diseases. "
                    "Always suggest consulting a doctor for medical advice. "
                    "Keep responses short and clear — max 3 sentences."
                )
            },
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.5,
        "max_tokens": 200
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"].strip()

    except Exception as e:
        print(f"⚠️  API call failed: {e}")
        return None


# ---------------- GRAPH NODES ---------------- #

# Node 1: Input Symptom
def input_node(state):
    state.symptom = input("\n🩺 Enter your symptom (or 'exit'): ").strip()
    return state


# Node 2: Observation Node
def observation_node(state):
    prompt = f"Give ONE short general possible cause or observation for this symptom: {state.symptom}"

    result = call_groq(prompt)
    observation = result if result else fallback_observation(state.symptom)

    state.observations.append(observation)
    state.steps_done += 1

    print(f"\n🔍 Observation {state.steps_done}: {observation}")
    return state


# Node 3: Analysis Node
def analysis_node(state):
    combined = " ".join(state.observations)
    prompt = f"Summarize these health observations in 2-3 sentences:\n{combined}"

    result = call_groq(prompt)
    state.analysis = result if result else "These observations suggest general possible causes but are not conclusive."

    print(f"\n🧠 Analysis:\n{state.analysis}")
    return state


# Node 4: Conditional Node (stop after 3 observations)
def condition_node(state):
    return len(state.observations) >= 3


# Node 5: Recommendation Node
def recommendation_node(state):
    prompt = f"""
Symptom: {state.symptom}
Analysis: {state.analysis}

Give a simple non-medical recommendation for this person.
Remind them to consult a doctor if needed.
Keep it under 3 sentences.
"""
    result = call_groq(prompt)
    state.recommendation = result if result else fallback_recommendation(state.symptom)
    return state


# Node 6: End Node
def end_node(state):
    print("\n" + "=" * 60)
    print("💡 Final Recommendation:")
    print(f"👉 {state.recommendation}")
    print("=" * 60)
    print("⚠️  Disclaimer: This is not medical advice. Always consult a qualified doctor.")
    return state


# ---------------- GRAPH EXECUTION ---------------- #
def run_symptom_tracker():
    print("=" * 60)
    print("   🩺 AI Symptom Tracker — Health Info Assistant")
    print("   Type 'exit' or 'quit' to end the session")
    print("=" * 60)

    state = SymptomState()

    while True:

        # Step 1: Take symptom input
        state = input_node(state)

        # Step 2: Exit check
        if state.symptom.lower() in ["exit", "quit", "bye"]:
            print("\n👋 Stay healthy! Goodbye.")
            print("=" * 60)
            break

        # Step 3: Empty input guard
        if not state.symptom:
            print("⚠️  Please enter a symptom.")
            continue

        # Step 4: Reset for new symptom session
        state.observations = []
        state.steps_done = 0
        state.analysis = ""
        state.recommendation = ""

        print(f"\n🔎 Analyzing: '{state.symptom}' ...")
        print("-" * 60)

        # Step 5: Loop — collect 3 observations + analysis
        while True:
            state = observation_node(state)
            state = analysis_node(state)

            if condition_node(state):
                break

        # Step 6: Final recommendation + output
        state = recommendation_node(state)
        state = end_node(state)


# ---------------- RUN ---------------- #
if __name__ == "__main__":
    run_symptom_tracker()

✅ Groq API key loaded.

   🩺 AI Symptom Tracker — Health Info Assistant
   Type 'exit' or 'quit' to end the session

🩺 Enter your symptom (or 'exit'): persistent cough

🔎 Analyzing: 'persistent cough' ...
------------------------------------------------------------

🔍 Observation 1: A persistent cough can be caused by a respiratory infection, such as bronchitis or pneumonia. It's best to consult a doctor to determine the underlying cause and receive proper treatment. A doctor's evaluation is necessary to rule out any underlying conditions that may be contributing to the cough.

🧠 Analysis:
A persistent cough can be caused by respiratory infections like bronchitis or pneumonia. To determine the underlying cause, it's best to consult a doctor for a proper evaluation. A doctor's assessment is necessary to identify and treat any contributing conditions.

🔍 Observation 2: A persistent cough can be caused by a respiratory infection, such as bronchitis or pneumonia. It's best to consult a doc

In [6]:
# ==========================================
# AI EMAIL WORKFLOW (HUMAN-IN-LOOP - FIXED)
# ==========================================

import requests
from google.colab import userdata
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

# 🔑 Load API Key from Colab Secrets
API_KEY = userdata.get('groq_api_key')

if not API_KEY:
    raise ValueError("❌ Add 'groq_api_key' in Colab Secrets (🔑 icon in sidebar).")

print("✅ Groq API key loaded.\n")


# ---------------- STATE ---------------- #
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool


# ---------------- GROQ CALL ---------------- #
def groq_call(prompt: str):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "llama-3.3-70b-versatile",   # ✅ Fixed: updated model
        "messages": [
            {
                "role": "system",
                "content": "You are a professional email writer. Write clear, polite, and formal emails."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.7,
        "max_tokens": 500
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"].strip()

    except Exception as e:
        print(f"⚠️  API call failed: {e}")
        return None


# ---------------- NODES ---------------- #

# Node 1: Draft Email
def draft_email(state: EmailState):
    print(f"\n⏳ Drafting email for: '{state['task']}' ...")
    print("-" * 60)

    prompt = f"""
Write a professional email about: {state['task']}.
Keep it clear, polite, and formal.
Include: Subject line, greeting, body, and sign-off.
"""

    draft = groq_call(prompt)

    if not draft:
        draft = (
            f"Subject: Regarding {state['task']}\n\n"
            f"Dear Recipient,\n\n"
            f"I am writing to you regarding {state['task']}. "
            f"Please let me know if you need any further information.\n\n"
            f"Best regards,\nSender"
        )

    print("\n📧 Draft Email:\n")
    print(draft)
    print("-" * 60)

    return {"draft": draft}


# Node 2: Human Review
def human_review(state: EmailState):
    print("\n🔍 Please review the draft email above.")

    while True:
        decision = input("\n👉 Approve this email? (yes/no): ").strip().lower()

        if decision in ["yes", "y"]:
            print("✅ You approved the email.")
            return {"approved": True}

        elif decision in ["no", "n"]:
            print("❌ You rejected the email.")
            return {"approved": False}

        else:
            print("⚠️  Invalid input. Please type 'yes' or 'no'.")


# Node 3: Send or Reject
def send_email(state: EmailState):
    print("\n" + "=" * 60)

    if state["approved"]:
        print("🚀 Email Approved & Sent Successfully!")
        print(f"📬 Task completed: {state['task']}")
    else:
        print("🗑️  Email Rejected. Draft discarded.")
        print(f"📌 Task cancelled: {state['task']}")

    print("=" * 60)

    return {"task": state["task"]}


# ---------------- GRAPH SETUP ---------------- #

builder = StateGraph(EmailState)

builder.add_node("draft",  draft_email)
builder.add_node("review", human_review)
builder.add_node("send",   send_email)

builder.set_entry_point("draft")

builder.add_edge("draft",  "review")
builder.add_edge("review", "send")
builder.add_edge("send",   END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)


# ---------------- RUN ---------------- #
def run_email_assistant():
    print("=" * 60)
    print("   📨 AI Email Assistant — Human-in-the-Loop")
    print("=" * 60)

    while True:
        task = input("\n📌 Enter email task (or 'exit' to quit): ").strip()

        if task.lower() in ["exit", "quit", "bye"]:
            print("\n👋 Goodbye!")
            break

        if not task:
            print("⚠️  Task cannot be empty. Please try again.")
            continue

        state = {
            "task": task,
            "draft": "",
            "approved": False
        }

        app.invoke(state, config={"configurable": {"thread_id": "email-session-1"}})

        print("\n🔄 Ready for next email task...")


# ---------------- MAIN ---------------- #
if __name__ == "__main__":
    run_email_assistant()

✅ Groq API key loaded.

   📨 AI Email Assistant — Human-in-the-Loop

📌 Enter email task (or 'exit' to quit): draf a email

⏳ Drafting email for: 'draf a email' ...
------------------------------------------------------------

📧 Draft Email:

Subject: Draft of Email for Review and Feedback

Dear Team,

I am writing to share with you a draft of an email that I have prepared for our upcoming marketing campaign. The email aims to promote our new product and services to our target audience, and I would like to request your review and feedback before it is finalized.

The draft email is attached to this message for your reference. I have included all the necessary details, including the product features, benefits, and a call to action. I believe that this email will effectively communicate our message and generate interest among our potential customers.

I would appreciate it if you could review the draft email and provide me with your feedback by the end of the week. Your input will be inva

Scenario Question
"Imagine you are designing an AI-powered assistant that drafts structured reports, pauses for human review, and then either publishes or rejects them. How would you structure the state and workflow graph to ensure accountability and human oversight in the process?"

In [8]:
# ==========================================
# AI REPORT WORKFLOW (HUMAN-IN-THE-LOOP)
# ==========================================

import requests
from google.colab import userdata
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

# 🔑 Load API Key from Colab Secrets
API_KEY = userdata.get('Groq_api')

if not API_KEY:
    raise ValueError("❌ Add 'groq_api_key' in Colab Secrets (🔑 icon in sidebar).")

print("✅ Groq API key loaded.\n")


# ---------------- STATE ---------------- #
class ReportState(TypedDict):
    task: str
    draft: str
    approved: bool


# ---------------- GROQ CALL ---------------- #
def groq_call(prompt: str):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "llama-3.3-70b-versatile",    # ✅ Fixed: updated model
        "messages": [
            {
                "role": "system",
                "content": "You are a professional report writer. Write clear, structured, and detailed reports."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.7,
        "max_tokens": 800
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"].strip()

    except Exception as e:
        print(f"⚠️  API call failed: {e}")
        return None


# ---------------- NODES ---------------- #

# Node 1: Draft Report
def draft_report(state: ReportState):
    print(f"\n⏳ Generating report for: '{state['task']}' ...")
    print("-" * 65)

    prompt = f"""
Generate a structured professional report on: {state['task']}.

Include the following sections:
1. Introduction
2. Key Points / Findings
3. Analysis
4. Conclusion
5. Recommendations

Make it detailed, clear, and professional.
"""

    draft = groq_call(prompt)

    if not draft:
        draft = (
            f"Report on: {state['task']}\n\n"
            f"1. Introduction\nThis report covers {state['task']}.\n\n"
            f"2. Key Points\n- Point 1\n- Point 2\n- Point 3\n\n"
            f"3. Conclusion\nFurther analysis is recommended.\n\n"
            f"4. Recommendations\nConsult relevant experts for detailed guidance."
        )

    print("\n📄 Draft Report:\n")
    print(draft)
    print("-" * 65)

    return {"draft": draft}


# Node 2: Human Review
def review_report(state: ReportState):
    print("\n🔍 Please review the report above.")

    while True:
        decision = input("\n👉 Approve this report? (yes/no): ").strip().lower()

        if decision in ["yes", "y"]:
            print("✅ You approved the report.")
            return {"approved": True}

        elif decision in ["no", "n"]:
            print("❌ You rejected the report.")
            return {"approved": False}

        else:
            print("⚠️  Invalid input. Please type 'yes' or 'no'.")


# Node 3: Publish or Reject
def publish_report(state: ReportState):
    print("\n" + "=" * 65)

    if state["approved"]:
        print("🚀 Report Approved & Published Successfully!")
        print(f"📬 Topic: {state['task']}")
    else:
        print("🗑️  Report Rejected. Draft discarded.")
        print(f"📌 Topic cancelled: {state['task']}")

    print("=" * 65)
    return {"task": state["task"]}


# ---------------- GRAPH SETUP ---------------- #

builder = StateGraph(ReportState)

builder.add_node("draft",   draft_report)
builder.add_node("review",  review_report)
builder.add_node("publish", publish_report)

builder.set_entry_point("draft")

builder.add_edge("draft",   "review")
builder.add_edge("review",  "publish")
builder.add_edge("publish", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)


# ---------------- RUN ---------------- #
def run_report_assistant():
    print("=" * 65)
    print("   📊 AI Report Assistant — Human-in-the-Loop")
    print("=" * 65)

    thread = 0  # thread counter for unique sessions

    while True:
        task = input("\n📌 Enter report topic (or 'exit' to quit): ").strip()

        if task.lower() in ["exit", "quit", "bye"]:
            print("\n👋 Goodbye!")
            break

        if not task:
            print("⚠️  Topic cannot be empty. Please try again.")
            continue

        thread += 1  # new unique thread per report

        state = {
            "task": task,
            "draft": "",
            "approved": False
        }

        # ✅ Fixed: pass thread_id in configurable
        app.invoke(
            state,
            config={"configurable": {"thread_id": str(thread)}}
        )

        print("\n🔄 Ready for next report...\n")


# ---------------- MAIN ---------------- #
if __name__ == "__main__":
    run_report_assistant()

✅ Groq API key loaded.

   📊 AI Report Assistant — Human-in-the-Loop

📌 Enter report topic (or 'exit' to quit): Web development

⏳ Generating report for: 'Web development' ...
-----------------------------------------------------------------

📄 Draft Report:

**Web Development Report**

**1. Introduction**

The world of web development has undergone significant transformations in recent years, driven by the rapid evolution of technology and the increasing demand for online presence. Web development encompasses a broad range of activities, from designing and building websites to creating complex web applications and mobile applications. This report aims to provide an in-depth analysis of the current state of web development, highlighting key trends, challenges, and opportunities. The purpose of this report is to inform stakeholders, including businesses, developers, and investors, about the latest developments in the field and provide recommendations for future growth and improvement.

